# 0417_2 · Sensibilidad S2 en validación con la base v4: regresión con todas las guías

**Para qué.** Iván pidió (P5, 25-10) ver la regresión entrenada con **todas** las guías que salen a reparto, también las que ya estaban vencidas en T0. Aquí se repite S2 con la base v4, los cinco bloques de validación (abril a agosto) y el modelo de la decisión v2 (LightGBM con pérdida L1). **No toca la reserva de septiembre y octubre.**

La sensibilidad S1 (hueco p99) ya no se corre: la ventana de 86,8 h quedó en el diseño principal de la v4 (0411_2).

**Modelos.** El de regresión de `P4b_decision_v2.json`, con sus hiperparámetros, y la regresión lineal como referencia. No hay búsqueda nueva.

**Regla de lectura (la misma de la v3).** S2 es un análisis secundario; el modelo principal sigue siendo la población con margen. Se reporta el MAE en todas, en las que tenían margen y en las vencidas, y la diferencia con la base sobre las mismas guías con margen (IC 95 % por días).

**Necesita:** 0411_2 a 0416_2 corridos (`V4/MODELADO/`, `pred_val_*.parquet`, `P4b_decision_v2.json`) y `MODULOS/` junto al cuaderno. **Escribe** solo tablas `P4c_*` agregadas en `V4/`.

In [ ]:
# Si faltan librerías:
# !pip install "duckdb<2" lightgbm xgboost scikit-learn

In [ ]:
%pip install "duckdb<2" lightgbm xgboost
dbutils.library.restartPython()

In [ ]:
import os, json, time, warnings
import numpy as np, pandas as pd, duckdb
from sklearn.metrics import average_precision_score, mean_absolute_error
warnings.filterwarnings("ignore", category=UserWarning)
from MODULOS.modelado_temporal import (cargar_preparacion, cargar_festivos, tabla_exclusiones, agregar_historia2,
                                       pliegues_origen_movil, partes_del_bloque, marco_predicciones, construir_modelo,
                                       predecir, tabla_metricas, diferencia_por_bloques)

PATH_DATABRICKS  = "/Volumes/proyectogrado/source/source/"
CARPETA_SALIDAS  = f"{PATH_DATABRICKS}LOG_CONSOLIDADO/REVISION_2026_11/V4/"
CARPETA_MODELADO = f"{CARPETA_SALIDAS}MODELADO/"
RUTA_DECISION    = f"{CARPETA_SALIDAS}P4b_decision_v2.json"                    # decisión v2 (0416_2)

USAR_PRED_GUARDADAS = True     # True: la base son las predicciones de validación de 0412_2/0415_2 (pred_val_*.parquet)
B_BOOTSTRAP         = 1000
EJECUTAR_S1, EJECUTAR_S2 = False, True   # S1 ya está en el diseño principal de la v4
try:
    display
except NameError:
    display = print

In [ ]:
# 0. Preparación de 0411 y decisión registrada
CFG, df, HIST = cargar_preparacion(CARPETA_MODELADO)
DEC = json.load(open(RUTA_DECISION))
SEMILLA, CATEGORICAS, TAREAS = CFG["SEMILLA"], CFG["CATEGORICAS"], CFG["TAREAS"]
NUMERICAS = DEC["variables_numericas"]
assert DEC["version"] == "internas", "Este cuaderno supone la versión 'internas' registrada en la decisión"
assert DEC.get("version_decision") == 2, "Se esperaba P4b_decision_v2.json"
MODELOS = {"reg": {DEC["modelo_reg"]: DEC["params_reg"], "Lineal": {}}}   # S2 solo usa la regresión
INICIOS = [pd.Timestamp(x) for x in CFG["INICIOS_VALIDACION"]]
print(f"{len(df):,} guías con margen · modelos: { {t: list(m) for t, m in MODELOS.items()} }")

In [ ]:
# Utilidades
def validar(d, hist, tarea, y_col, pliegues, etiqueta, filtro_tr=None):
    '''Entrena y predice cada bloque con los modelos de la decisión. filtro_tr(ini) -> máscara booleana sobre d
    que restringe además el entrenamiento (S1). Devuelve la tabla de predicciones.'''
    out = []
    for ini, tr, va in pliegues:
        if filtro_tr is not None:
            tr = tr[filtro_tr(ini)[tr]]
        Dva, Xtr, ytr, Xva, yva = partes_del_bloque(d, hist[ini], y_col, tr, va, CATEGORICAS + NUMERICAS)
        for nombre, params in MODELOS[tarea].items():
            m = construir_modelo(tarea, nombre, NUMERICAS, CATEGORICAS, params, SEMILLA).fit(Xtr, ytr)
            out.append(marco_predicciones(etiqueta, tarea, ini, Dva, yva).assign(modelo=nombre, pred=predecir(m, Xva, tarea),
                                                                              n_entrenamiento=len(ytr)))
        print(f"{etiqueta} · {tarea} · {ini.date()}: entrenamiento {len(ytr):,} · validación {len(yva):,}")
    return pd.concat(out, ignore_index=True)

def comparar(a, b, tarea, etiqueta):
    '''Diferencia de la métrica (a − b) sobre las mismas guías, con IC 95 % por días.'''
    met = average_precision_score if tarea == "clf" else mean_absolute_error
    m = a.merge(b[["bloque", "guia", "modelo", "pred"]], on=["bloque", "guia", "modelo"], suffixes=("", "_b"))
    filas = []
    for modelo, g in m.groupby("modelo"):
        filas.append({"comparacion": etiqueta, "tarea": tarea, "modelo": modelo, "metrica": "AP" if tarea == "clf" else "MAE_h",
                      "guias": len(g), **diferencia_por_bloques(g.y.values, g.pred.values, g.pred_b.values, g.dia.values,
                                                                met, B=B_BOOTSTRAP, semilla=SEMILLA)})
    return filas

PLIEGUES = {t: list(pliegues_origen_movil(df, CFG["INICIOS_VALIDACION"], CFG["FIN_DESARROLLO"], t_etiqueta=te))
            for t, _, te in TAREAS}

In [ ]:
# 1. Base: predicciones de validación de 0416 (o reentrenadas aquí con la misma configuración)
if USAR_PRED_GUARDADAS:
    p = pd.concat([pd.read_parquet(f"{CARPETA_MODELADO}pred_val_{m}.parquet", filters=[("version", "==", "internas"), ("tarea", "==", "reg")])
                   for m in MODELOS["reg"]], ignore_index=True)
    base = p[p.modelo.isin(list(MODELOS["reg"]))].copy()      # pred_val_LightGBM guarda la configuración elegida (L1)
    base["version"] = "base"
else:
    base = pd.concat([validar(df, HIST, t, y, PLIEGUES[t], "base") for t, y, _ in TAREAS if t == "reg"], ignore_index=True)
base["bloque"] = pd.to_datetime(base.bloque).dt.date
display(tabla_metricas(base, ["version", "tarea", "modelo"]).round(4))

## S2 · Regresión con todas las guías que salen a reparto

Se arma la población como en 0411 **sin** el filtro «No vencido antes de T0» (los demás filtros iguales), se recalcula la historia sin futuro con esa población y se valida la regresión. Las métricas se reportan en tres grupos: todas, con margen en T0 y vencidas en T0.

In [ ]:
# S2.a Población ampliada (mismos pasos y variables derivadas que 0411), cargada con poca memoria:
#      solo las columnas que usan los modelos, solo el desarrollo (T0 antes de FIN_DESARROLLO) y filtros en DuckDB.
if EJECUTAR_S2:
    import gc
    base = base[base.tarea == "reg"].copy()                      # S2 solo usa la regresión
    for nombre in ["df", "HIST", "PLIEGUES", "s1", "previo", "horas"]:
        globals().pop(nombre, None)
    gc.collect()
    cols_cat = [c for c in CATEGORICAS if c not in ("franja", "dia_semana")]
    cols_num = [c for c in NUMERICAS if not c.startswith(("hist_", "n_hist")) and c not in ("es_festivo", "volumen_declarado_cm3")]
    columnas = sorted(set(["guia", "t0", "fecha_entrega", "t_etiqueta_reg", "t_etiqueta_clf", "h_ultima_milla",
                           "incumplimiento_sla", "vencido_antes_de_t0", "LARGO_ENVIO", "ANCHO_ENVIO", "ALTO_ENVIO"]
                          + cols_cat + cols_num + (["ID_CLIENTE"] if CFG["USAR_VARIABLES_CLIENTE"] else [])))
    SLA_PRINCIPAL = ["SLA_UNICO"] if CFG["SOLO_SLA_UNICO"] else ["SLA_UNICO", "SLA_POR_REGLA"]
    filtros = [("SLA principal", "origen_sla IN (" + ", ".join(f"'{x}'" for x in SLA_PRINCIPAL) + ")"),
               ("Sin devolución ni siniestro", "estado_desenlace NOT IN ('DEVUELTO', 'SINIESTRADO')"),
               ("Entrega no anterior a la admisión (A2)", "estado_desenlace <> 'ENTREGA_ANTES_DE_ADMISION'"),
               (f"T0 al menos {CFG['P99_T0_A_ENTREGA_H']} h antes del corte (C1)", f"t0 <= TIMESTAMP '{CFG['FIN_EVALUABLE']}'")]
    if CFG["EXCLUIR_ENTREGA_SIN_HORA"]:
        filtros.append(("Entrega con hora (no 00:00:00)", "fecha_entrega IS NULL OR fecha_entrega <> date_trunc('day', fecha_entrega)"))
    if CFG["usa_clima"]:
        filtros.append(("Clima completo", "COALESCE(CLIMA_COMPLETO, FALSE)"))
    G2 = f"read_parquet('{CFG['ruta_gold']}')"
    acumulado, filas = "TRUE", []
    for nombre, cond in filtros:                                  # tabla de exclusiones, solo conteos
        acumulado = f"{acumulado} AND ({cond})"
        filas.append({"paso": nombre, "guias": duckdb.sql(f"SELECT COUNT(*) FROM {G2} WHERE {acumulado}").fetchall()[0][0]})
    excl2 = pd.DataFrame(filas)
    d2 = duckdb.sql(f"SELECT {', '.join(columnas)} FROM {G2} WHERE {acumulado} ORDER BY t0").df()
    for c in d2.select_dtypes("float64").columns:
        d2[c] = d2[c].astype("float32")
    neg = d2.h_ultima_milla < 0
    d2.loc[neg, ["h_ultima_milla", "t_etiqueta_reg"]] = np.nan
    sin_hora = d2.fecha_entrega.notna() & (d2.fecha_entrega == d2.fecha_entrega.dt.normalize())
    d2.loc[sin_hora, ["h_ultima_milla", "t_etiqueta_reg"]] = np.nan   # A1: sin hora no entran a la regresión
    festivos = cargar_festivos(CFG["RUTA_CALENDARIO"])
    d2["franja"] = pd.cut(d2.t0.dt.hour, [-1, 5, 11, 17, 23], labels=["MADRUGADA", "MANANA", "TARDE", "NOCHE"]).astype(str)
    d2["dia_semana"] = d2.t0.dt.dayofweek.astype(str)
    d2["zona_franja"] = d2["categoria_zona"].astype(str) + "_" + d2["franja"]
    d2["dia"] = d2.t0.dt.date
    d2["es_festivo"] = d2.t0.dt.normalize().isin(festivos).astype("int8")
    vol = d2.LARGO_ENVIO * d2.ANCHO_ENVIO * d2.ALTO_ENVIO
    d2["volumen_declarado_cm3"] = vol.where(vol > 0)
    d2 = d2.drop(columns=["LARGO_ENVIO", "ANCHO_ENVIO", "ALTO_ENVIO"]); del vol
    if CFG["USAR_VARIABLES_CLIENTE"]:
        d2["ID_CLIENTE"] = d2["ID_CLIENTE"].astype(str)
    for c in CATEGORICAS: d2[c] = d2[c].astype(str)
    d2["vencida_en_t0"] = d2.vencido_antes_de_t0.fillna(False).astype(bool)
    gc.collect()
    display(excl2)
    print(f"Población ampliada (desarrollo, base v4): {len(d2):,} guías · vencidas en T0: {d2.vencida_en_t0.mean():.1%} · "
          f"memoria {d2.memory_usage(deep=True).sum() / 1e9:.1f} GB")

In [ ]:
# S2.b Historia sin futuro con la población ampliada y validación de la regresión
if EJECUTAR_S2:
    t_ini = time.time()
    HIST2 = {ini: agregar_historia2(d2, ini, CFG["USAR_VARIABLES_CLIENTE"]) for ini in INICIOS}
    te_reg = dict((t, te) for t, _, te in TAREAS)["reg"]
    pl2 = list(pliegues_origen_movil(d2, CFG["INICIOS_VALIDACION"], CFG["FIN_DESARROLLO"], t_etiqueta=te_reg))
    s2 = validar(d2, HIST2, "reg", "h_ultima_milla", pl2, "S2_reg_todas")
    s2["bloque"] = pd.to_datetime(s2.bloque).dt.date
    s2 = s2.merge(d2[["guia", "vencida_en_t0"]], on="guia", how="left")
    del HIST2; gc.collect()
    s2["grupo"] = np.where(s2.vencida_en_t0, "vencidas en T0", "con margen en T0")
    print(f"S2: {time.time() - t_ini:.0f} s")

In [ ]:
# S2.c Métricas por grupo y comparación con la base sobre las mismas guías con margen
if EJECUTAR_S2:
    tabla_s2 = pd.concat([tabla_metricas(s2.assign(grupo="todas"), ["version", "grupo", "tarea", "modelo"]),
                          tabla_metricas(s2, ["version", "grupo", "tarea", "modelo"]),
                          tabla_metricas(base[base.tarea == "reg"].assign(grupo="con margen en T0"),
                                         ["version", "grupo", "tarea", "modelo"])], ignore_index=True).round(4)
    con_margen = s2[s2.grupo == "con margen en T0"]
    dif_s2 = pd.DataFrame(comparar(con_margen, base[base.tarea == "reg"], "reg",
                                   "S2 (entrenado con todas) − base, en guías con margen")).round(4)
    tabla_s2.to_csv(f"{CARPETA_SALIDAS}P4c_S2_regresion_todas_metricas.csv", index=False)
    dif_s2.to_csv(f"{CARPETA_SALIDAS}P4c_S2_regresion_todas_diferencias.csv", index=False)
    excl2.to_csv(f"{CARPETA_SALIDAS}P4c_S2_exclusiones.csv", index=False)
    display(tabla_s2); display(dif_s2)

In [ ]:
# 3. Lectura (S2 es secundario; S1 ya no aplica en la v4)
if EJECUTAR_S2:
    print("S2 (secundario):")
    display(tabla_s2[tabla_s2.modelo == DEC["modelo_reg"]][["version", "grupo", "n", "MAE_h", "MAE_10pct_mas_lento_h"]])